# Train a DDPM on resting-state EEG epochs (\*.fif)

This notebook mirrors the original NER example but is tailored to epoched resting-state EEG stored as MNE `Epochs`.
It relies on the `FIFDataLoader`, which ingests `.fif` files exported from your preprocessing pipeline and can optionally
condition on subject IDs or diagnosis labels.


## Prepare your epochs as .fif files

* Export each subject's preprocessed epochs to a standalone `.fif` file (e.g., using `epochs.save('CLASS_SUBJECTID_epochs.fif')`).
* If you want class conditioning, follow the `CLASS_SUBJECTID_*.fif` naming convention (e.g., `DEMENTIA_S001_epochs.fif`).
* Keep epoch lengths consistent across files; the loader will skip files that do not match the inferred `sfreq`, `n_channels`, or `n_times`.


In [ ]:
# Example: exporting an MNE Epochs object for one subject
# epochs: mne.Epochs = ...  # your preprocessed epochs
# class_label = 'DEMENTIA'
# subject_id = 'S001'
# epochs.save(f'/path/to/epochs_dir/{class_label}_{subject_id}_epochs.fif', overwrite=True)


In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
from hydra import compose, initialize
from omegaconf import OmegaConf
from pandas import read_csv

from ntd.datasets import FIFDataLoader
from ntd.train_diffusion_model import training_and_eval_pipeline
from ntd.utils.plotting_utils import (
    basic_plotting,
    plot_overlapping_signal,
    plot_sd,
)
from ntd.utils.utils import l2_distances

logging.basicConfig(level=logging.INFO)


## Optional: generate a tiny synthetic .fif dataset for testingThis cell writes 20 fake EEG files (5 per class) with 20 epochs each, 19 channels, 5-second epochs at 200 Hz.

In [ ]:
from pathlib import Path

import mne
import numpy as np

data_path = Path('../data/fake_rseeg').resolve()
data_path.mkdir(parents=True, exist_ok=True)

sfreq = 200  # Hz
n_channels = 19
n_times = 1000  # 5 seconds * 200 Hz
n_epochs = 20
class_labels = ['hc', 'smc', 'mci', 'dementia']

ch_names = [f'Ch{i:02d}' for i in range(1, n_channels + 1)]
info = mne.create_info(ch_names, sfreq=sfreq, ch_types='eeg')
rng = np.random.default_rng(0)

for label in class_labels:
    for subj_idx in range(5):
        # Simulate small-amplitude activity in Volts
        data = rng.standard_normal((n_epochs, n_channels, n_times)) * 1e-6
        events = np.column_stack(
            (
                np.arange(0, n_epochs * n_times, n_times),
                np.zeros(n_epochs, dtype=int),
                np.ones(n_epochs, dtype=int),
            )
        )
        epochs = mne.EpochsArray(
            data, info, events=events, event_id={'rest': 1}, tmin=0, verbose=False
        )
        fname = data_path / f"{label}_sub-{subj_idx:04d}_epochs.fif"
        epochs.save(fname, overwrite=True, verbose=False)

print(f"Wrote {len(list(data_path.glob('*.fif')))} files to {data_path}")


## Create the config for rSEEG

Override the config to point to your `.fif` files and to match your data dimensions.
Set `condition_on_class_label` to `True` if you want to condition on diagnosis (and adjust `network.cond_dim`).


In [ ]:
data_path = '../data/fake_rseeg'  # directory containing your exported or synthetic .fif files
condition_on_class_label = True
condition_on_subject_id = False

with initialize(version_base=None, config_path='../conf'):
    cfg = compose(
        config_name='config',
        overrides=[
            'dataset=fif_data_example',
            f'dataset.file_path={data_path}',
            f'dataset.condition_on_class_label={condition_on_class_label}',
            f'dataset.condition_on_subject_id={condition_on_subject_id}',
            'base.tag=rseeg_unconditional',
            'base.wandb_mode=disabled',
            'base.save_path=null',
            'optimizer.num_epochs=1000',
            'optimizer.lr=0.0004',
            'optimizer.train_batch_size=16',
            'network.signal_channel=19',
            'network.in_kernel_size=65',
            'network.out_kernel_size=65',
            'network.slconv_kernel_size=65',
            'network.num_scales=1',
            'network.hidden_channel=32',
            'network.num_off_diag=32',
            'network.use_pos_emb=True',
            '+experiments/generate_samples=generate_samples',
        ],
    )

# Peek at the dataset once to infer dimensions and adjust cond_dim / signal_length
preview_dataset = FIFDataLoader(
    file_path=cfg.dataset.file_path,
    n_epochs=cfg.dataset.n_epochs,
    condition_on_class_label=cfg.dataset.condition_on_class_label,
    condition_on_subject_id=cfg.dataset.condition_on_subject_id,
    verbose=True,
)

cfg.dataset.signal_length = preview_dataset.n_times
cfg.network.signal_channel = preview_dataset.n_channels
if preview_dataset.cond is not None:
    cfg.network.cond_dim = preview_dataset.cond_dim
else:
    cfg.network.cond_dim = 0

# Generate as many samples as we have training epochs by default
cfg.generate_samples.num_samples = len(preview_dataset)

print(OmegaConf.to_yaml(cfg))


## Train the model and generate samples


In [ ]:
diffusion_model, samples = training_and_eval_pipeline(cfg)
samples_numpy = samples.numpy()


In [ ]:
# Load the same dataset for evaluation/plotting
rseeg_dataset = FIFDataLoader(
    file_path=cfg.dataset.file_path,
    n_epochs=cfg.dataset.n_epochs,
    condition_on_class_label=cfg.dataset.condition_on_class_label,
    condition_on_subject_id=cfg.dataset.condition_on_subject_id,
)
raw_numpy = rseeg_dataset.data_array_np

assert samples_numpy.shape[1:] == raw_numpy.shape[1:]
num_trials, num_channels, sig_length = raw_numpy.shape
fs = rseeg_dataset.sfreq


## Plot some random real and generated samples


In [ ]:
rand_id = np.random.randint(len(samples_numpy))
print(rand_id)
offset = -1.1
fig, ax = plt.subplots()
plot_overlapping_signal(
    fig,
    ax,
    sig=raw_numpy[rand_id] + offset * np.arange(num_channels)[:, np.newaxis],
    colors=['dimgrey'],
)
basic_plotting(
    fig,
    ax,
    y_ticks=[],
    x_lim=(0, sig_length),
    x_ticks=[0, sig_length],
    x_ticklabels=[0, sig_length / fs],
    x_label='time [s]',
)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots()
plot_overlapping_signal(
    fig,
    ax,
    samples_numpy[rand_id] + offset * np.arange(num_channels)[:, np.newaxis],
    colors=['black'],
)
basic_plotting(
    fig,
    ax,
    y_ticks=[],
    x_lim=(0, sig_length),
    x_ticks=[0, sig_length],
    x_ticklabels=[0, sig_length / fs],
    x_label='time [s]',
)
fig.tight_layout()
plt.show()


## Plot the power spectral density

Red is generated, black is real. Pointwise median and 25% / 75% percentiles are shown.


In [ ]:
fig, axs = plt.subplots(num_channels // 10 + 1, 10, figsize=(45, 25))
for idx in range(num_channels):
    plot_sd(
        fig=fig,
        ax=axs[idx // 10, idx % 10],
        arr_one=raw_numpy[:, idx, :],
        arr_two=samples_numpy[:, idx, :],
        fs=fs,
        nperseg=sig_length,
        agg_function=np.median,
        with_quantiles=True,
        x_ss=slice(0, int(fs // 2)),
        color_one='black',
        color_two='C3',
    )
plt.show()


## Plot the evoked potentials

For all channels. Red is generated, black is real. Mean and standard deviation are shown.


In [ ]:
fig, axs = plt.subplots(num_channels // 5 + 1, 5, figsize=(45, 45))
time_axis = np.arange(sig_length) / fs
for idx in range(num_channels):
    axs[idx // 5, idx % 5].fill_between(
        time_axis,
        np.quantile(raw_numpy[:, idx, :], 0.1, axis=0),
        np.quantile(raw_numpy[:, idx, :], 0.9, axis=0),
        color='black',
        alpha=0.2,
    )
    axs[idx // 5, idx % 5].fill_between(
        time_axis,
        np.quantile(samples_numpy[:, idx, :], 0.1, axis=0),
        np.quantile(samples_numpy[:, idx, :], 0.9, axis=0),
        color='C3',
        alpha=0.2,
    )
    axs[idx // 5, idx % 5].plot(
        time_axis,
        np.mean(raw_numpy[:, idx, :], axis=0),
        color='black',
    )
    axs[idx // 5, idx % 5].plot(
        time_axis,
        np.mean(samples_numpy[:, idx, :], axis=0),
        color='C3',
    )
plt.show()


## Plot the topomaps

This example reuses channel locations from your `.fif` files (or a custom CSV).


In [ ]:
# Option 1: read channel names/montage from the first .fif file
first_fif = sorted(Path(cfg.dataset.file_path).glob('*.fif'))[0]
epochs_info = mne.read_epochs(first_fif, preload=False, verbose='WARNING').info
info = mne.create_info(
    ch_names=epochs_info.ch_names, sfreq=fs, ch_types='eeg'
)
info.set_montage(epochs_info.get_montage())

# Option 2: replace the block above with a CSV like in the original example
# csv_path = '../data/CorrectedChannelsLocation.csv'
# chan_info = read_csv(csv_path)
# montage = mne.channels.make_standard_montage('standard_1020')
# info = mne.create_info(ch_names=list(chan_info['Labels']), sfreq=fs, ch_types='eeg')
# info.set_montage(montage)

times = np.linspace(0, sig_length / fs, num=5, endpoint=False)
for samples in [raw_numpy, samples_numpy]:
    evoked = mne.EvokedArray(samples.mean(0), info)
    evoked.plot_topomap(
        times,
        ch_type='eeg',
        scalings=1.0,
        vlim=(-0.5, 0.5),
        image_interp='cubic',
        colorbar=False,
        res=300,
        size=1.5,
    )
